# Ответы на вопросы

## 1. Аналитическое решение задачи регрессии (матричная форма)

### Постановка задачи

Пусть у нас есть матрица обьектов-признаков $𝑋∈𝑅^{𝑁×𝐷}$ (где N - число обьектов, D - число признаков), вектор целевых значений $y∈𝑅^{N}$ и вектор весов $w∈𝑅^{D}$

Модель линейной регрессии:

$$𝑦=𝑋𝑤$$

Замечание: Свободный член (bias) 
𝑏
 можно учесть, добавив к матрице 
𝑋
 единичный столбец. Тогда 
𝑏
 войдёт в вектор 
𝑤
 как одна из компонент.

### Функция потерь (MSE/RSS)

В матричной форме: 

$$L(w)=\frac{1}{N}||y- Xw||^{2}_2=\frac{1}{N}(y-Xw)^{T}(y-Xw)$$

Раскроем скобки (константу 1/𝑁 можно опустить, она не влияет на точку минимума):

$$𝐿(𝑤)=𝑦^𝑇𝑦−2𝑤^𝑇𝑋^𝑇𝑦+𝑤^𝑇𝑋^𝑇𝑋𝑤$$

### Дифференцирование и поиск минимума

Берём градиент по w и приравниваем к нулю:

$$\frac{∂𝐿}{∂𝑤}=−2𝑋^{𝑇}𝑦+2𝑋^{𝑇}𝑋𝑤=0$$

Это так называемые нормальные уравнения. Решая относительно 𝑤:

$$𝑤=(𝑋^𝑇𝑋)^{−1}𝑋^{𝑇}𝑦$$


## 2. Изменения в решении при добавлении L1 и L2 регуляризации


Функция потерь при L2-регуляризации (Ridge):

$$𝐿(𝑤)=∥𝑦−𝑋𝑤∥^{2}_2+𝜆∥𝑤∥^2$$

Функция потерь при L1-регуляризации (Lasso):

$$𝐿(𝑤)=∥𝑦−𝑋𝑤∥^{2}_2+𝜆∥𝑤∥_1$$

## 3. Почему L1-регуляризация используется для отбора признаков

L1-регуляризация зануляет многие веса, превращая модель в разреженную (sparse). Это позволяет автоматически отбирать только значимые признаки.

## 4. Как сделать линейные модели нелинейными

Линейные модели (Linear Regression, Ridge, Lasso) линейны по параметрам 𝑤, но не обязательно по признакам. Это позволяет моделировать нелинейные зависимости.

Идея: замена признаков на нелинейные функции

Заменяем входной вектор 𝑥 на вектор нелинейных преобразований 𝜙(𝑥):

$$𝑓(𝑥;𝑤)=𝑤^𝑇𝜙(𝑥)$$

Примеры преобразований 𝜙(𝑥)

1. Полиномиальные признаки
2. Тригонометрические признаки (для периодических данных)
3. Логарифмические / экспоненциальные преобразования отдельных признаков.

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.linear_model import LinearRegression as SkLinearRegression
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures

In [2]:
train = pd.read_json('data/train.json')
test = pd.read_json('data/test.json')

## Task 3. Intro data analysis part 2

In [3]:
def clean_value(value):
    for ch in ['[', ']', "'", '"']:
        value = value.replace(ch, '')
    return value

def clean_features_column(df):
    df['features'] = df['features'].apply(
        lambda lst: [clean_value(f) for f in lst if clean_value(f)]
    )
    return df

train = clean_features_column(train)
test = clean_features_column(test)

huge_list = []

for index, row in train.iterrows():
    for feature in row['features']:
        huge_list.append(feature)

In [4]:
unique_values = set(huge_list)
print(f'Количество уникальных значений: {len(unique_values)}')

Количество уникальных значений: 1554


In [5]:
counter = Counter(huge_list)

top_20 = counter.most_common(20)

top_20_features = [name for name, count in top_20]
print(top_20)

[('Elevator', 25915), ('Cats Allowed', 23540), ('Hardwood Floors', 23527), ('Dogs Allowed', 22035), ('Doorman', 20898), ('Dishwasher', 20426), ('No Fee', 18062), ('Laundry in Building', 16344), ('Fitness Center', 13252), ('Pre-War', 9148), ('Laundry in Unit', 8738), ('Roof Deck', 6542), ('Outdoor Space', 5268), ('Dining Room', 5136), ('High Speed Internet', 4299), ('Balcony', 2992), ('Swimming Pool', 2730), ('Laundry In Building', 2593), ('New Construction', 2559), ('Terrace', 2283)]


In [6]:
def make_binary_features(df, top_features):
    feature_sets = df['features'].apply(set)
    for feature in top_features:
        df[feature] = feature_sets.apply(
            lambda s: 1 if feature in s else 0
        )
    return df

train = make_binary_features(train, top_20_features)
test = make_binary_features(test, top_20_features)

feature_list = top_20_features + ['bathrooms', 'bedrooms']

print(feature_list)

['Elevator', 'Cats Allowed', 'Hardwood Floors', 'Dogs Allowed', 'Doorman', 'Dishwasher', 'No Fee', 'Laundry in Building', 'Fitness Center', 'Pre-War', 'Laundry in Unit', 'Roof Deck', 'Outdoor Space', 'Dining Room', 'High Speed Internet', 'Balcony', 'Swimming Pool', 'Laundry In Building', 'New Construction', 'Terrace', 'bathrooms', 'bedrooms']


In [7]:
lower = train['price'].quantile(0.01)
upper = train['price'].quantile(0.99)

train = train[(train['price'] >= lower) & (train['price'] <= upper)]
train = train[train['bathrooms'] <= 20]

test = test[(test['price'] >= lower) & (test['price'] <= upper)]
test = test[test['bathrooms'] <= 20]

In [8]:
X_train = train[feature_list]
X_test = test[feature_list]

y_train = train['price'].values
y_test = test['price'].values

X_train_np = X_train.values.astype(float)
X_test_np = X_test.values.astype(float)

## Task 4. Models implementation - Linear regression

In [9]:
class LinearRegression:
    def __init__(self, method='sgd', learning_rate=0.01, n_epochs=100,
                 batch_size = 1, random_state=42, fit_intercept=True, normalize=True):
        self.method = method
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.random_state = random_state
        self.fit_intercept = fit_intercept
        self.normalize = normalize
        self.weights = None
        self.bias = 0.0
        self._mu = None
        self._sigma = None

    def _scale(self, X, fit=False):
        if not self.normalize:
            return X
        if fit:
            self._mu = X.mean(axis=0)
            self._sigma = X.std(axis=0)
            self._sigma[self._sigma == 0] = 1.0
        return (X - self._mu) / self._sigma

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        X = self._scale(X, fit=True)

        if self.method == 'normal':
            self._fit_normal(X, y)
        elif self.method == 'gd':
            self._fit_gd(X, y)
        elif self.method == 'sgd':
            self._fit_sgd(X, y)
        else:
            raise ValueError("method must be 'normal', 'gd' or 'sgd'")
        return self

    def _fit_normal(self, X, y):
        if self.fit_intercept:
            X = np.hstack([np.ones((X.shape[0], 1)), X])
        w = np.linalg.pinv(X.T @ X) @ X.T @ y
        if self.fit_intercept:
            self.bias = w[0]
            self.weights = w[1:]
        else:
            self.bias = 0.0
            self.weights = w

    def _fit_gd(self, X, y):
        n, m = X.shape
        self.weights = np.zeros(m)
        self.bias = 0.0
        for _ in range(self.n_epochs):
            y_pred = X @ self.weights + self.bias
            error = y_pred - y
            grad_w = (2 / n) * (X.T @ error)
            grad_b = (2 / n) * np.sum(error)
            self.weights -= self.learning_rate * grad_w
            self.bias -= self.learning_rate * grad_b

    def _fit_sgd(self, X, y):
        n, m = X.shape
        self.weights = np.zeros(m)
        self.bias = 0.0
        rng = np.random.RandomState(self.random_state)

        for epoch in range(self.n_epochs):
            indices = rng.permutation(n)
            lr = self.learning_rate / (1 + 0.01 * epoch)
            for start in range(0, n, self.batch_size):
                batch_idx = indices[start:start + self.batch_size]
                X_b, y_b = X[batch_idx], y[batch_idx]
                error = (X_b @ self.weights + self.bias) - y_b
                b = len(batch_idx)
                self.weights -= lr * (2 / b) * (X_b.T @ error)
                self.bias    -= lr * (2 / b) * np.sum(error)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        X = self._scale(X, fit=False)
        return X @ self.weights + self.bias

In [10]:
def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

def mae(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

In [11]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te, results):
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)
    results.append({
        'model': name,
        'mae_train': mae(y_tr, pred_tr), 'mae_test': mae(y_te, pred_te),
        'rmse_train': rmse(y_tr, pred_tr), 'rmse_test': rmse(y_te, pred_te),
        'r2_train': r2_score(y_tr, pred_tr), 'r2_test': r2_score(y_te, pred_te),
    })

results = []

lr_normal = LinearRegression(method='normal').fit(X_train_np, y_train)
evaluate('Normal', lr_normal, X_train_np, y_train, X_test_np, y_test, results)

lr_gd = LinearRegression(method='gd', learning_rate=0.1, n_epochs=500).fit(X_train_np, y_train)
evaluate('GD', lr_gd, X_train_np, y_train, X_test_np, y_test, results)


lr_sgd = LinearRegression(method='sgd', learning_rate=0.05, n_epochs=500,
                          batch_size=64, random_state=42).fit(X_train_np, y_train)
evaluate('SGD', lr_sgd, X_train_np, y_train, X_test_np, y_test, results)

In [12]:
sk = SkLinearRegression()
sk.fit(X_train_np, y_train)
evaluate('Sklearn', sk, X_train_np, y_train, X_test_np, y_test, results)

In [13]:
res_df = pd.DataFrame(results)

mae_table = res_df[['model', 'mae_train', 'mae_test']].rename(
    columns={'mae_train': 'train', 'mae_test': 'test'})

rmse_table = res_df[['model', 'rmse_train', 'rmse_test']].rename(
    columns={'rmse_train': 'train', 'rmse_test': 'test'})

r2_table = res_df[['model', 'r2_train', 'r2_test']].rename(
    columns={'r2_train': 'train', 'r2_test': 'test'})

print("=== MAE ===")
print(mae_table.to_string(index=False))
print("\n=== RMSE ===")
print(rmse_table.to_string(index=False))
print("\n=== R2 ===")
print(r2_table.to_string(index=False))

=== MAE ===
  model      train       test
 Normal 711.791166 714.679694
     GD 711.791160 714.679764
    SGD 715.449462 718.760705
Sklearn 711.791166 714.679694

=== RMSE ===
  model       train        test
 Normal 1035.351576 1043.464696
     GD 1035.351576 1043.464658
    SGD 1038.053386 1047.698809
Sklearn 1035.351576 1043.464696

=== R2 ===
  model    train     test
 Normal 0.580034 0.569521
     GD 0.580034 0.569522
    SGD 0.577839 0.566021
Sklearn 0.580034 0.569521


## Task 5. Regularized models implementation - Ridge, Lasso, ElasticNet

In [14]:
def soft_threshold(rho, lam):
    if rho < -lam:
        return rho + lam
    elif rho > lam:
        return rho- lam
    else:
        return 0.0

class RegularizedRegression:
    def __init__(self, penalty='l2', alpha=1.0, l1_ratio=0.5, n_iter=1000, tol=1e-6, fit_intercept=True):
        self.penalty = penalty
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.n_iter = n_iter
        self.tol = tol
        self.fit_intercept = fit_intercept
        self.weights = None
        self.bias = 0.0
        pass

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        if self.penalty == 'l2':
            self._fit_ridge(X, y)
        else:
            self._fit_coordinate_descent(X, y)
        return self
    
    def _fit_ridge(self, X, y):
        n, m = X.shape
        if self.fit_intercept:
            Xb = np.hstack([np.ones((n, 1)), X])
        else:
            Xb = X
        m_full = Xb.shape[1]

        I = np.eye(m_full)
        if self.fit_intercept:
            I[0, 0] = 0.0
        
        w = np.linalg.solve(Xb.T @ Xb + self.alpha * I, Xb.T @ y)

        if self.fit_intercept:
            self.bias = w[0]
            self.weights = w[1:]
        else:
            self.bias = 0.0
            self.weights = w
    
    def _fit_coordinate_descent(self, X, y):
        n, m = X.shape

        if self.fit_intercept:
            X_mean = X.mean(axis=0)
            y_mean = y.mean()
            Xc = X - X_mean
            yc = y - y_mean
        else:
            Xc, yc = X, y
            X_mean, y_mean = np.zeros(m), 0.0
        
        w = np.zeros(m)

        if self.penalty == 'l1':
            l1 = self.alpha
            l2 = 0.0
        else:
            l1 = self.alpha * self.l1_ratio
            l2 = self.alpha * (1 - self.l1_ratio)
        
        col_sq = np.sum(Xc ** 2, axis=0)

        for it in range(self.n_iter):
            w_old = w.copy()
            for j in range(m):
                residual = yc - Xc @ w + Xc[:, j] * w[j]
                rho = Xc[:, j] @ residual

                if col_sq[j] == 0:
                    w[j] = 0.0
                    continue
                w[j] = soft_threshold(rho, l1 * n) / (col_sq[j] + l2 * n)

            if np.max(np.abs(w - w_old)) < self.tol:
                break
        
        self.weights = w
        self.bias = y_mean - X_mean @ w if self.fit_intercept else 0.0

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.weights + self.bias


In [15]:
X_train_s = X_train_np
X_test_s = X_test_np

In [16]:
ALPHA = 0.01
L1_RATIO = 0.5

results_reg = []

my_ridge = RegularizedRegression(penalty='l2', alpha=ALPHA).fit(X_train_s, y_train)
evaluate('My Ridge', my_ridge, X_train_np, y_train, X_test_np, y_test, results_reg)

my_lasso = RegularizedRegression(penalty='l1', alpha=ALPHA, n_iter=2000).fit(X_train_s, y_train)
evaluate('My Lasso', my_lasso, X_train_np, y_train, X_test_np, y_test, results_reg)

my_enet = RegularizedRegression(penalty='elasticnet', alpha=ALPHA,
                                l1_ratio=L1_RATIO, n_iter=2000).fit(X_train_s, y_train)
evaluate('My ElasticNet', my_enet, X_train_np, y_train, X_test_np, y_test, results_reg)

sk_ridge = Ridge(alpha=ALPHA).fit(X_train_s, y_train)
evaluate('Sklearn Ridge', sk_ridge, X_train_np, y_train, X_test_np, y_test, results_reg)

sk_lasso = Lasso(alpha=ALPHA, max_iter=10000).fit(X_train_s, y_train)
evaluate('Sklearn Lasso', sk_lasso, X_train_np, y_train, X_test_np, y_test, results_reg)

sk_enet = ElasticNet(alpha=ALPHA, l1_ratio=L1_RATIO, max_iter=10000).fit(X_train_s, y_train)
evaluate('Sklearn ElasticNet', sk_enet, X_train_np, y_train, X_test_np, y_test, results_reg)

In [17]:
res_df = pd.DataFrame(results_reg)

mae_table  = res_df[['model','mae_train','mae_test']].rename(columns={'mae_train':'train','mae_test':'test'})
rmse_table = res_df[['model','rmse_train','rmse_test']].rename(columns={'rmse_train':'train','rmse_test':'test'})
r2_table   = res_df[['model','r2_train','r2_test']].rename(columns={'r2_train':'train','r2_test':'test'})

print("=== MAE ===");  print(mae_table.to_string(index=False))
print("\n=== RMSE ==="); print(rmse_table.to_string(index=False))
print("\n=== R2 ===");   print(r2_table.to_string(index=False))

=== MAE ===
             model      train       test
          My Ridge 711.791133 714.679658
          My Lasso 711.784875 714.673069
     My ElasticNet 711.281260 714.042769
     Sklearn Ridge 711.791133 714.679658
     Sklearn Lasso 711.784876 714.673073
Sklearn ElasticNet 711.281264 714.042774

=== RMSE ===
             model       train        test
          My Ridge 1035.351576 1043.464672
          My Lasso 1035.351595 1043.463125
     My ElasticNet 1035.623714 1043.195367
     Sklearn Ridge 1035.351576 1043.464672
     Sklearn Lasso 1035.351595 1043.463123
Sklearn ElasticNet 1035.623716 1043.195369

=== R2 ===
             model    train     test
          My Ridge 0.580034 0.569521
          My Lasso 0.580034 0.569523
     My ElasticNet 0.579813 0.569744
     Sklearn Ridge 0.580034 0.569521
     Sklearn Lasso 0.580034 0.569523
Sklearn ElasticNet 0.579813 0.569744


## Task 6. Feature normalization

### 1. Нормализация данных: когда и зачем

Нормализация ОБЯЗАТЕЛЬНА

| Ситуация | Причина |
|----------|---------|
| Градиентный спуск (линейная/логистическая регрессия, нейросети) | Признаки в разных масштабах делают поверхность потерь вытянутой → медленная/нестабильная сходимость. Масштабирование резко ускоряет процесс. |
| Регуляризованные модели (Ridge, Lasso, ElasticNet) | Штраф $\alpha \sum w_j^2$ действует на величину весов. Без масштабирования признаки с большим диапазоном значений штрафуются несправедливо сильно. |
| Методы, основанные на расстояниях (KNN, K-Means, SVM с RBF-ядром) | Они вычисляют евклидово расстояние. Признак с диапазоном 0–100000 полностью подавляет признак с диапазоном 0–1, искажая понятие «близости». |

Нормализация НЕ НУЖНА (или бесполезна)

| Ситуация | Причина |
|----------|---------|
| Деревья решений (Decision Tree, Random Forest, Gradient Boosting / XGBoost) | Они разделяют данные по порогам ($x > t$). Монотонное масштабирование не меняет порядок разбиения → эффекта нет. |
| Признаки уже сравнимы | Например, если все признаки — бинарные (0/1), масштабирование почти ничего не даёт. |
| Наивный Байес (Naive Bayes) | Работает с вероятностями и распределениями, а не с величинами. |

### 2. MinMaxScaler — формула

Масштабирует каждый признак в диапазон **[0, 1]**:

$$X_{\text{scaled}} = \frac{X - X_{\min}}{X_{\max} - X_{\min}}$$

### Произвольный диапазон $[a, b]$

Обобщённая формула для масштабирования в любой заданный интервал:

$$X_{\text{scaled}} = a + \frac{(X - X_{\min}) \cdot (b - a)}{X_{\max} - X_{\min}}$$

### Важные замечания

- **$\min$ и $\max$ вычисляются по каждому столбцу отдельно**
- **Только на обучающей выборке!** Тестовые данные масштабируются с использованием этих же сохранённых значений
- Метод **сохраняет форму распределения**, но **чувствителен к выбросам**:
  - Одно экстремальное значение может «растянуть» весь диапазон
  - В этом случае лучше использовать `RobustScaler` (основанный на медиане и квартилях)

In [18]:
class MyMinMaxScaler:
    def __init__(self, feature_range=(0,1)):
        self._range = feature_range
        self.data_min_ = None
        self.data_max_ = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.data_min_ = X.min(axis=0)
        self.data_max_ = X.max(axis=0)
        self.range_ = self.data_max_ - self.data_min_
        self.range_[self.range_ == 0] = 1.0
        return self
    
    def transform(self, X):
        X = np.asarray(X, dtype=float)
        a, b = self._range
        X_std = (X - self.data_min_) / self.range_
        return X_std * (b - a) + a
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [19]:
sk_mm = MinMaxScaler()
X_train_sk = sk_mm.fit_transform(X_train_np)
X_test_sk = sk_mm.transform(X_test_np)


In [20]:
my_mm = MyMinMaxScaler()
X_train_my = my_mm.fit_transform(X_train_np)
X_test_my = my_mm.transform(X_test_np)

print("MinMax — max abs diff (train):", np.max(np.abs(X_train_my - X_train_sk)))
print("MinMax — max abs diff (test): ", np.max(np.abs(X_test_my  - X_test_sk)))
print("Allclose:", np.allclose(X_train_my, X_train_sk) and
                    np.allclose(X_test_my,  X_test_sk))

MinMax — max abs diff (train): 5.551115123125783e-17
MinMax — max abs diff (test):  1.1102230246251565e-16
Allclose: True


### StandardScaler — формула

StandardScaler выполняет стандартизацию признаков: приводит их к нулевому среднему и единичному стандартному отклонению.

Формула
$$X_{scaled}=\frac{X−μ}{σ}$$

где: μ — среднее значение по столбцу, σ — стандартное отклонение по столбцу

Оба параметра (μ и σ) вычисляются только на обучающей выборке, а затем применяются к тестовым данным.


In [21]:
class MyStandardScaler:
    def __init__(self):
        self.mean_ = None
        self.scale_ = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0)
        self.scale_[self.scale_ == 0] = 1.0
        return self
    
    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.mean_) / self.scale_
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [22]:
sk_ss = StandardScaler()
X_train_sk_s = sk_ss.fit_transform(X_train_np)
X_test_sk_s = sk_ss.transform(X_test_np)

my_ss = MyStandardScaler()
X_train_my_s = my_ss.fit_transform(X_train_np)
X_test_my_s = my_ss.transform(X_test_np)

print("Standard — max abs diff (train):", np.max(np.abs(X_train_my_s - X_train_sk_s)))
print("Standard — max abs diff (test): ", np.max(np.abs(X_test_my_s  - X_test_sk_s)))
print("Allclose:", np.allclose(X_train_my_s, X_train_sk_s) and
                    np.allclose(X_test_my_s,  X_test_sk_s))

# sanity check: scaled train should have mean≈0, std≈1
print("mean ≈", np.round(X_train_my_s.mean(axis=0)[:3], 6))
print("std  ≈", np.round(X_train_my_s.std(axis=0)[:3], 6))

Standard — max abs diff (train): 0.0
Standard — max abs diff (test):  0.0
Allclose: True
mean ≈ [-0.  0.  0.]
std  ≈ [1. 1. 1.]


### Task 7. Fit custom and sklearn models with normalized data

In [23]:
ALPHA = 0.01
L1_RATIO = 0.5

results_norm = []

def fit_all_models(X_tr, X_te, y_tr, y_te, scaler_name, results):
    lr_normal = LinearRegression(method='normal', normalize=False).fit(X_tr, y_tr)
    evaluate(f'[{scaler_name}] My Linear (normal)', lr_normal, X_tr, y_tr, X_te, y_te, results)

    lr_sgd = LinearRegression(method='sgd', learning_rate=0.05, n_epochs=500,
                              batch_size=64, random_state=42, normalize=False).fit(X_tr, y_tr)
    evaluate(f'[{scaler_name}] My Linear (sgd)', lr_sgd, X_tr, y_tr, X_te, y_te, results)

    my_ridge = RegularizedRegression(penalty='l2', alpha=ALPHA).fit(X_tr, y_tr)
    evaluate(f'[{scaler_name}] My Ridge', my_ridge, X_tr, y_tr, X_te, y_te, results)

    my_lasso = RegularizedRegression(penalty='l1', alpha=ALPHA, n_iter=2000).fit(X_tr, y_tr)
    evaluate(f'[{scaler_name}] My Lasso', my_lasso, X_tr, y_tr, X_te, y_te, results)

    my_enet = RegularizedRegression(penalty='elasticnet', alpha=ALPHA,
                                    l1_ratio=L1_RATIO, n_iter=2000).fit(X_tr, y_tr)
    evaluate(f'[{scaler_name}] My ElasticNet', my_enet, X_tr, y_tr, X_te, y_te, results)

    sk_lr = SkLinearRegression().fit(X_tr, y_tr)
    evaluate(f'[{scaler_name}] Sk Linear', sk_lr, X_tr, y_tr, X_te, y_te, results)

    sk_ridge = Ridge(alpha=ALPHA).fit(X_tr, y_tr)
    evaluate(f'[{scaler_name}] Sk Ridge', sk_ridge, X_tr, y_tr, X_te, y_te, results)

    sk_lasso = Lasso(alpha=ALPHA, max_iter=10000).fit(X_tr, y_tr)
    evaluate(f'[{scaler_name}] Sk Lasso', sk_lasso, X_tr, y_tr, X_te, y_te, results)

    sk_enet = ElasticNet(alpha=ALPHA, l1_ratio=L1_RATIO, max_iter=10000).fit(X_tr, y_tr)
    evaluate(f'[{scaler_name}] Sk ElasticNet', sk_enet, X_tr, y_tr, X_te, y_te, results) 

fit_all_models(X_train_my, X_test_my, y_train, y_test, 'MinMax', results_norm)

fit_all_models(X_train_my_s, X_test_my_s, y_train, y_test, 'Standard', results_norm)   

In [24]:
res_df_norm = pd.DataFrame(results_norm)

mae_table  = res_df_norm[['model', 'mae_train', 'mae_test']].rename(
    columns={'mae_train': 'train', 'mae_test': 'test'})
rmse_table = res_df_norm[['model', 'rmse_train', 'rmse_test']].rename(
    columns={'rmse_train': 'train', 'rmse_test': 'test'})
r2_table   = res_df_norm[['model', 'r2_train', 'r2_test']].rename(
    columns={'r2_train': 'train', 'r2_test': 'test'})

print("=== MAE ===");   print(mae_table.to_string(index=False))
print("\n=== RMSE ==="); print(rmse_table.to_string(index=False))
print("\n=== R2 ===");   print(r2_table.to_string(index=False))

=== MAE ===
                        model      train       test
  [MinMax] My Linear (normal) 711.791166 714.679694
     [MinMax] My Linear (sgd) 714.052926 716.945970
            [MinMax] My Ridge 711.791749 714.679829
            [MinMax] My Lasso 711.786377 714.673357
       [MinMax] My ElasticNet 772.124894 773.491473
           [MinMax] Sk Linear 711.791166 714.679694
            [MinMax] Sk Ridge 711.791749 714.679829
            [MinMax] Sk Lasso 711.786378 714.673361
       [MinMax] Sk ElasticNet 772.124896 773.491478
[Standard] My Linear (normal) 711.791166 714.679694
   [Standard] My Linear (sgd) 715.449462 718.760705
          [Standard] My Ridge 711.791160 714.679687
          [Standard] My Lasso 711.788871 714.677396
     [Standard] My ElasticNet 711.663174 714.528050
         [Standard] Sk Linear 711.791166 714.679694
          [Standard] Sk Ridge 711.791160 714.679687
          [Standard] Sk Lasso 711.788871 714.677398
     [Standard] Sk ElasticNet 711.663175 714.528051


In [25]:
base_features = ['bathrooms', 'bedrooms']

X_train_base = train[base_features].values.astype(float)
X_test_base  = test[base_features].values.astype(float)

poly = PolynomialFeatures(degree=10, include_bias=False)
X_train_poly = poly.fit_transform(X_train_base)
X_test_poly  = poly.transform(X_test_base)


poly_scaler = MyStandardScaler()
X_train_poly_s = poly_scaler.fit_transform(X_train_poly)
X_test_poly_s  = poly_scaler.transform(X_test_poly)

results_poly = []

lr_poly = LinearRegression(method='normal', normalize=False).fit(X_train_poly_s, y_train)
evaluate('Linreg Polynomial', lr_poly, X_train_poly_s, y_train, X_test_poly_s, y_test, results_poly)

ridge_poly = RegularizedRegression(penalty='l2', alpha=ALPHA).fit(X_train_poly_s, y_train)
evaluate('Ridge Polynomial', ridge_poly, X_train_poly_s, y_train, X_test_poly_s, y_test, results_poly)

lasso_poly = RegularizedRegression(penalty='l1', alpha=ALPHA, n_iter=2000).fit(X_train_poly_s, y_train)
evaluate('Lasso Polynomial', lasso_poly, X_train_poly_s, y_train, X_test_poly_s, y_test, results_poly)

enet_poly = RegularizedRegression(penalty='elasticnet', alpha=ALPHA, l1_ratio=L1_RATIO,
                                  n_iter=2000).fit(X_train_poly_s, y_train)
evaluate('ElasticNet Polynomial', enet_poly, X_train_poly_s, y_train, X_test_poly_s, y_test, results_poly)

poly_df = pd.DataFrame(results_poly)
print("\n=== Polynomial models ===")
print(poly_df[['model','mae_train','mae_test','rmse_train','rmse_test','r2_train','r2_test']].to_string(index=False))


=== Polynomial models ===
                model  mae_train     mae_test  rmse_train    rmse_test  r2_train       r2_test
    Linreg Polynomial 756.283124 2.740174e+07 1077.608481 5.492800e+09  0.545053 -1.192844e+13
     Ridge Polynomial 757.472622 2.629118e+04 1080.132572 5.251677e+06  0.542920 -1.090415e+07
     Lasso Polynomial 761.671824 4.602765e+03 1087.207781 7.345419e+05  0.536912 -2.133178e+05
ElasticNet Polynomial 766.149652 2.426900e+03 1095.695000 3.207425e+05  0.529654 -4.067226e+04


In [26]:
class NaiveModel:
    def __init__(self, strategy='mean'):
        self.strategy = strategy
        self.value = None

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        if self.strategy == 'mean':
            self.value = np.mean(y)
        elif self.strategy == 'median':
            self.value = np.median(y)
        else:
            raise ValueError("strategy must be 'mean' or 'median'")
        return self

    def predict(self, X):
        X = np.asarray(X)
        return np.full(X.shape[0], self.value)

results_naive = []

naive_mean = NaiveModel(strategy='mean').fit(X_train_np, y_train)
evaluate('Naive mean', naive_mean, X_train_np, y_train, X_test_np, y_test, results_naive)

naive_median = NaiveModel(strategy='median').fit(X_train_np, y_train)
evaluate('Naive median', naive_median, X_train_np, y_train, X_test_np, y_test, results_naive)

naive_df = pd.DataFrame(results_naive)
print("=== Naive models ===")
print(naive_df[['model','mae_train','mae_test','rmse_train','rmse_test','r2_train','r2_test']].to_string(index=False))

=== Naive models ===
       model   mae_train    mae_test  rmse_train   rmse_test  r2_train   r2_test
  Naive mean 1139.192515 1137.845295 1597.646655 1590.401945  0.000000 -0.000023
Naive median 1086.210505 1084.595285 1644.236439 1635.406116 -0.059173 -0.057419


In [ ]:
results_default = []
fit_all_models(X_train_np, X_test_np, y_train, y_test, 'default', results_default)

def rename_models(results, mapping):
    for r in results:
        for old, new in mapping.items():
            if old in r['model']:
                r['model'] = r['model'].replace(old, new)
    return results

all_results = []
all_results.extend(results_default)
all_results.extend(results_norm)
all_results.extend(results_poly)
all_results.extend(results_naive)

final_df = pd.DataFrame(all_results)

result_MAE = final_df[['model', 'mae_train', 'mae_test']].rename(
    columns={'mae_train': 'train', 'mae_test': 'test'})
result_RMSE = final_df[['model', 'rmse_train', 'rmse_test']].rename(
    columns={'rmse_train': 'train', 'rmse_test': 'test'})
result_R2 = final_df[['model', 'r2_train', 'r2_test']].rename(
    columns={'r2_train': 'train', 'r2_test': 'test'})

print("=" * 60)
print("=== FINAL MAE ===")
print(result_MAE.to_string(index=False))
print("\n=== FINAL RMSE ===")
print(result_RMSE.to_string(index=False))
print("\n=== FINAL R2 ===")
print(result_R2.to_string(index=False))

=== FINAL MAE ===
                        model       train         test
 [default] My Linear (normal)  711.791166 7.146797e+02
    [default] My Linear (sgd)  716.587576 7.197267e+02
           [default] My Ridge  711.791133 7.146797e+02
           [default] My Lasso  711.784875 7.146731e+02
      [default] My ElasticNet  711.281260 7.140428e+02
          [default] Sk Linear  711.791166 7.146797e+02
           [default] Sk Ridge  711.791133 7.146797e+02
           [default] Sk Lasso  711.784876 7.146731e+02
      [default] Sk ElasticNet  711.281264 7.140428e+02
  [MinMax] My Linear (normal)  711.791166 7.146797e+02
     [MinMax] My Linear (sgd)  714.052926 7.169460e+02
            [MinMax] My Ridge  711.791749 7.146798e+02
            [MinMax] My Lasso  711.786377 7.146734e+02
       [MinMax] My ElasticNet  772.124894 7.734915e+02
           [MinMax] Sk Linear  711.791166 7.146797e+02
            [MinMax] Sk Ridge  711.791749 7.146798e+02
            [MinMax] Sk Lasso  711.786378 7.146

In [30]:
## Bonus 11.c - Linear regression with batch / mini-batch / analytical

# В классе LinearRegression уже реализованы:
#   method='normal' - аналитическое решение
#   method='gd'     - полный батч (batch gradient descent)
#   method='sgd' c batch_size=1   - стохастический (SGD)
#   method='sgd' c batch_size=N   - mini-batch

results_batch = []

# Аналитическое решение
m1 = LinearRegression(method='normal', normalize=True).fit(X_train_np, y_train)
evaluate('Analytical (normal)', m1, X_train_np, y_train, X_test_np, y_test, results_batch)

# Full batch GD
m2 = LinearRegression(method='gd', learning_rate=0.1, n_epochs=1000, normalize=True).fit(X_train_np, y_train)
evaluate('Batch GD', m2, X_train_np, y_train, X_test_np, y_test, results_batch)

# Mini-batch SGD
m3 = LinearRegression(method='sgd', learning_rate=0.05, n_epochs=500,
                      batch_size=64, normalize=True).fit(X_train_np, y_train)
evaluate('Mini-batch SGD (bs=64)', m3, X_train_np, y_train, X_test_np, y_test, results_batch)

# Pure SGD (batch_size=1)
m4 = LinearRegression(method='sgd', learning_rate=0.01, n_epochs=100,
                      batch_size=1, normalize=True).fit(X_train_np, y_train)
evaluate('SGD (bs=1)', m4, X_train_np, y_train, X_test_np, y_test, results_batch)

batch_df = pd.DataFrame(results_batch)
print("=== Different training methods ===")
print(batch_df[['model','mae_test','rmse_test','r2_train','r2_test']].to_string(index=False))

=== Different training methods ===
                 model   mae_test   rmse_test  r2_train  r2_test
   Analytical (normal) 714.679694 1043.464696  0.580034 0.569521
              Batch GD 714.679694 1043.464695  0.580034 0.569521
Mini-batch SGD (bs=64) 718.760705 1047.698809  0.577839 0.566021
            SGD (bs=1) 825.004778 1154.618213  0.486873 0.472925
